**Plan**  
Assume input structure as a flattened array.  
Implement CNN models later once you are able to combine bands from all data sources along the same location.

# Baselines

In [1]:
def get_features_labels_flat():
    """
    Filler for feature and label pipeline input
    """
    from sklearn.datasets import load_iris
    data = load_iris()
    X, y = data["data"], data["target"]

    return X, y

X, y = get_features_labels_flat()
X.shape, y.shape

((150, 4), (150,))

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import numpy as np
np.random.seed(25)

def model_logistic_regression(X, y):
    def train_log_reg(X_train, y_train):
        """
        log reg assumes flattened inputs
        """
        model = LogisticRegression(max_iter=250)
        model.fit(X_train, y_train)

        return model
        
    def log_reg_accuracy(model, X_test, y_test, is_binary=True):
        """
        Binary log-reg
        """
        preds = model.predict(X_test)
        preds = np.round(preds) 

        metrics = dict()
        metrics["overall"] = accuracy_score(y_test, preds)
        
        average = "binary" if is_binary else "weighted"
        metrics["f1"] = f1_score(y_test, preds, average=average)
        metrics["recall"] = recall_score(y_test, preds, average=average)
        metrics["precision"] = precision_score(y_test, preds, average=average)

        return metrics


    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)

    # print(X_train.shape, y_train.shape)
    logreg = train_log_reg(X_train, y_train)

    train_metrics = log_reg_accuracy(logreg, X_train, y_train, is_binary=False)
    test_metrics = log_reg_accuracy(logreg, X_test, y_test, is_binary=False)
    print(f"Train metrics:\n{train_metrics}")
    print(f"Test metrics:\n{test_metrics}")

model_logistic_regression(X, y)

Train metrics:
{'overall': 0.9833333333333333, 'f1': 0.9832956503014643, 'recall': 0.9833333333333333, 'precision': 0.9840909090909091}
Test metrics:
{'overall': 0.9666666666666667, 'f1': 0.9669803921568627, 'recall': 0.9666666666666667, 'precision': 0.9703703703703703}


In [3]:
import cv2
import numpy as np
import pandas as pd

def get_features_labels_cnn():
    """
    Filler for feature and label pipeline input
    """
    from sklearn.datasets import fetch_openml
    mnist = fetch_openml('mnist_784', version=1)
    X, y = mnist["data"], mnist["target"]
    X, y = X.to_numpy().astype(float), y.to_numpy().astype(float)

    return X, y

X, y = get_features_labels_cnn()
X.shape, y.shape

((70000, 784), (70000,))

In [4]:
model_logistic_regression(X,y)

Train metrics:
{'overall': 0.9392321428571428, 'f1': 0.939173259724209, 'recall': 0.9392321428571428, 'precision': 0.9391789503563871}
Test metrics:
{'overall': 0.9173571428571429, 'f1': 0.917085437341133, 'recall': 0.9173571428571429, 'precision': 0.9170960266034655}


c:\Users\seani\miniforge3\envs\modeling_pipeline\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
import torch

unet = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet',
    in_channels=3, out_channels=1, init_features=32, pretrained=True)

X = X[...,np.newaxis].repeat(3, axis=-1)
X.shape

Using cache found in C:\Users\seani/.cache\torch\hub\mateuszbuda_brain-segmentation-pytorch_master


(70000, 784, 3)

In [ ]:
import numpy as np
from PIL import Image
from torchvision.transforms import v2

m, s = np.mean(X, axis=(0, 1)), np.std(X, axis=(0, 1))
preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize(mean=m, std=s),
    v2.ToDtype(torch.float32, scale=True)
])

input_tensor = preprocess(X)
input_batch = input_tensor.unsqueeze(0)

if torch.cuda.is_available():
    input_batch = input_batch.to('cuda')
    unet = unet.to('cuda')

with torch.no_grad():
    output = unet(input_batch)

print(torch.round(output[0]))

c:\Users\seani\miniforge3\envs\modeling_pipeline\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
